# Notebook 05 — Coverage Phase Diagram

This notebook synthesizes prior results from residue-manifold learning experiments:

- Notebook 03: NMF lane recovery
- Notebook 04: SAE dilution / fragmentation

The goal is to compare methods under shared metrics:

- lane coverage
- lane-mass alignment
- reconstruction error
- redundancy
- dead features
- structure quality

Figures are saved as **SVG only**. One figure = one canonical format.

## 1. Setup

In [ ]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

mpl.rcParams["svg.fonttype"] = "none"
mpl.rcParams["figure.dpi"] = 120

os.makedirs("data", exist_ok=True)
os.makedirs("figures", exist_ok=True)

VALID_LANES_MOD30 = [1, 7, 11, 13, 17, 19, 23, 29]
N_LANES = len(VALID_LANES_MOD30)


def save_svg(fig, name):
    path = f"figures/{name}.svg"
    fig.savefig(path, bbox_inches="tight")
    print(f"Saved: {path}")

## 2. Locate required CSV files

Notebook 05 expects prior summary data. This regenerated version searches `data/` for likely files and accepts common filename variants.

In [ ]:
def list_csvs(folder="data"):
    if not os.path.exists(folder):
        return []
    return sorted([f for f in os.listdir(folder) if f.lower().endswith(".csv")])

csv_files = list_csvs("data")
print("CSV files in data/:")
for f in csv_files:
    print(" -", f)

In [ ]:
def score_file(filename, include_terms, exclude_terms=None):
    """Score a filename by included/excluded keyword matches."""
    exclude_terms = exclude_terms or []
    name = filename.lower()
    score = 0
    for term in include_terms:
        if term.lower() in name:
            score += 1
    for term in exclude_terms:
        if term.lower() in name:
            score -= 2
    return score


def find_best_csv(include_terms, exclude_terms=None, required=True):
    scored = []
    for f in csv_files:
        s = score_file(f, include_terms, exclude_terms)
        if s > 0:
            scored.append((s, f))
    scored.sort(reverse=True)
    if scored:
        return scored[0][1]
    if required:
        raise FileNotFoundError(
            f"Could not find CSV matching {include_terms}. Available files: {csv_files}"
        )
    return None

# Prefer exact summary outputs, but tolerate common variants.
nmf_file = None
sae_file = None

if "nmf_recovery_summary.csv" in csv_files:
    nmf_file = "nmf_recovery_summary.csv"
else:
    nmf_file = find_best_csv(["nmf", "summary"], exclude_terms=["component", "feature"], required=False)
    if nmf_file is None:
        nmf_file = find_best_csv(["nmf", "recovery"], exclude_terms=["component", "feature"], required=True)

if "sae_dilution_summary.csv" in csv_files:
    sae_file = "sae_dilution_summary.csv"
else:
    sae_file = find_best_csv(["sae", "summary"], exclude_terms=["feature", "representative", "comparison"], required=False)
    if sae_file is None:
        sae_file = find_best_csv(["sae", "dilution"], exclude_terms=["feature", "representative", "comparison"], required=True)

print("
Using files:")
print("NMF:", f"data/{nmf_file}")
print("SAE:", f"data/{sae_file}")

nmf_raw = pd.read_csv(f"data/{nmf_file}")
sae_raw = pd.read_csv(f"data/{sae_file}")

print("
NMF columns:", list(nmf_raw.columns))
print("SAE columns:", list(sae_raw.columns))

## 3. Normalize schemas

Prior notebooks may use slightly different column names. This section standardizes them for shared comparison.

In [ ]:
def first_existing(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None


def normalize_nmf(nmf):
    out = pd.DataFrame()
    cap_col = first_existing(nmf, ["capacity", "k", "n_components", "components"])
    mse_col = first_existing(nmf, ["reconstruction_mse", "mse", "final_loss", "loss"])
    lane_col = first_existing(nmf, ["mean_lane_mass_ratio", "lane_mass_ratio", "alignment", "mean_alignment"])
    coverage_col = first_existing(nmf, ["coverage", "lane_coverage"])
    dead_col = first_existing(nmf, ["dead_features"])
    red_col = first_existing(nmf, ["redundant_valid_features", "redundancy", "redundant_features"])

    if cap_col is None:
        raise ValueError(f"NMF summary needs a capacity column. Columns: {list(nmf.columns)}")
    if mse_col is None:
        raise ValueError(f"NMF summary needs reconstruction_mse/final_loss column. Columns: {list(nmf.columns)}")

    out["method"] = "NMF"
    out["capacity"] = nmf[cap_col].astype(float)
    out["topk"] = np.nan
    out["reconstruction_mse"] = nmf[mse_col].astype(float)
    out["lane_mass_ratio"] = nmf[lane_col].astype(float) if lane_col else 1.0
    out["coverage"] = nmf[coverage_col].astype(float) if coverage_col else 1.0
    out["dead_features"] = nmf[dead_col].astype(float) if dead_col else 0.0
    out["redundant_valid_features"] = nmf[red_col].astype(float) if red_col else np.maximum(out["capacity"] - N_LANES, 0)
    return out


def normalize_sae(sae):
    out = pd.DataFrame()
    cap_col = first_existing(sae, ["capacity", "hidden_dim", "dictionary_size", "n_features", "features"])
    topk_col = first_existing(sae, ["topk", "top_k", "k_sparse"])
    mse_col = first_existing(sae, ["reconstruction_mse", "final_loss", "mse", "loss"])
    coverage_col = first_existing(sae, ["coverage", "lane_coverage"])
    lane_col = first_existing(sae, ["mean_lane_mass_ratio", "lane_mass_ratio", "alignment", "mean_alignment"])
    dead_col = first_existing(sae, ["dead_features"])
    red_col = first_existing(sae, ["redundant_valid_features", "redundancy", "redundant_features"])

    required = {
        "capacity": cap_col,
        "reconstruction_mse/final_loss": mse_col,
        "coverage": coverage_col,
        "lane_mass_ratio": lane_col,
    }
    missing = [name for name, col in required.items() if col is None]
    if missing:
        raise ValueError(f"SAE summary missing {missing}. Columns: {list(sae.columns)}")

    out["method"] = "SAE"
    out["capacity"] = sae[cap_col].astype(float)
    out["topk"] = sae[topk_col].astype(float) if topk_col else np.nan
    out["reconstruction_mse"] = sae[mse_col].astype(float)
    out["coverage"] = sae[coverage_col].astype(float)
    out["lane_mass_ratio"] = sae[lane_col].astype(float)
    out["dead_features"] = sae[dead_col].astype(float) if dead_col else 0.0
    out["redundant_valid_features"] = sae[red_col].astype(float) if red_col else 0.0
    return out

nmf = normalize_nmf(nmf_raw)
sae = normalize_sae(sae_raw)

df = pd.concat([nmf, sae], ignore_index=True)

df.head()

## 4. Define provisional structure quality

This score is intentionally simple and interpretable. It is **not yet CGCS**; it is a practical bridge metric for phase-diagram comparison.

High quality requires:

- high coverage
- high lane alignment
- low redundancy
- low dead-feature count

In [ ]:
df["structure_quality"] = (
    df["coverage"]
    * df["lane_mass_ratio"]
    * (1.0 / (1.0 + df["redundant_valid_features"]))
    * (1.0 / (1.0 + df["dead_features"]))
)


def classify(row):
    if row["coverage"] >= 0.99 and row["lane_mass_ratio"] >= 0.99 and row["dead_features"] == 0:
        return "recovered"
    if row["coverage"] < 0.75:
        return "fragmented"
    if row["dead_features"] > 0 or row["redundant_valid_features"] > 0:
        return "diluted"
    return "partial"


df["regime"] = df.apply(classify, axis=1)

df.to_csv("data/coverage_phase_diagram.csv", index=False)
print("Saved: data/coverage_phase_diagram.csv")

df

## 5. Figure — Coverage phase diagram

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

nmf_plot = df[df["method"] == "NMF"].sort_values("capacity")
ax.plot(nmf_plot["capacity"], nmf_plot["coverage"], marker="o", label="NMF")

sae_plot = df[df["method"] == "SAE"].copy()
for topk, group in sae_plot.groupby("topk", dropna=False):
    group = group.sort_values("capacity")
    label = f"SAE top-k={int(topk)}" if pd.notna(topk) else "SAE"
    ax.plot(group["capacity"], group["coverage"], marker="o", label=label)

ax.axhline(1.0, linestyle="--", linewidth=1, alpha=0.5)
ax.set_xlabel("Capacity (components / features)")
ax.set_ylabel("Lane coverage")
ax.set_title("Coverage phase diagram")
ax.set_ylim(-0.05, 1.05)
ax.legend()
fig.tight_layout()
save_svg(fig, "coverage_phase_diagram")
plt.show()

## 6. Figure — Alignment phase diagram

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(nmf_plot["capacity"], nmf_plot["lane_mass_ratio"], marker="o", label="NMF")

for topk, group in sae_plot.groupby("topk", dropna=False):
    group = group.sort_values("capacity")
    label = f"SAE top-k={int(topk)}" if pd.notna(topk) else "SAE"
    ax.plot(group["capacity"], group["lane_mass_ratio"], marker="o", label=label)

ax.axhline(1.0, linestyle="--", linewidth=1, alpha=0.5)
ax.set_xlabel("Capacity (components / features)")
ax.set_ylabel("Lane-mass ratio")
ax.set_title("Alignment phase diagram")
ax.set_ylim(-0.05, 1.05)
ax.legend()
fig.tight_layout()
save_svg(fig, "alignment_phase_diagram")
plt.show()

## 7. Figure — Cost vs structure quality

This plot separates reconstruction cost from structural interpretability.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

for method, group in df.groupby("method"):
    ax.scatter(
        group["reconstruction_mse"],
        group["structure_quality"],
        s=40 + 4 * group["capacity"],
        alpha=0.8,
        label=method,
    )
    for _, row in group.iterrows():
        label = f"{int(row['capacity'])}"
        if row["method"] == "SAE" and pd.notna(row["topk"]):
            label = f"{int(row['capacity'])}/k{int(row['topk'])}"
        ax.annotate(label, (row["reconstruction_mse"], row["structure_quality"]), fontsize=8, alpha=0.75)

ax.set_xlabel("Reconstruction MSE")
ax.set_ylabel("Structure quality")
ax.set_title("Reconstruction cost vs structure quality")
ax.legend()
fig.tight_layout()
save_svg(fig, "cost_vs_structure_quality")
plt.show()

## 8. Figure — Method regime map

In [ ]:
regime_order = ["recovered", "partial", "diluted", "fragmented"]
regime_to_y = {r: i for i, r in enumerate(regime_order)}

df["regime_y"] = df["regime"].map(regime_to_y)

fig, ax = plt.subplots(figsize=(8, 5))

for method, group in df.groupby("method"):
    ax.scatter(
        group["capacity"],
        group["regime_y"],
        s=60,
        alpha=0.8,
        label=method,
    )

for _, row in df.iterrows():
    label = ""
    if row["method"] == "SAE" and pd.notna(row["topk"]):
        label = f"k{int(row['topk'])}"
    else:
        label = "NMF"
    ax.annotate(label, (row["capacity"], row["regime_y"]), fontsize=8, alpha=0.75, xytext=(4, 3), textcoords="offset points")

ax.set_xlabel("Capacity (components / features)")
ax.set_ylabel("Regime")
ax.set_yticks(range(len(regime_order)))
ax.set_yticklabels(regime_order)
ax.set_title("Method regime map")
ax.legend()
fig.tight_layout()
save_svg(fig, "method_regime_map")
plt.show()

## 9. Method summary table

In [ ]:
method_summary = (
    df.groupby("method")
    .agg(
        best_structure_quality=("structure_quality", "max"),
        best_coverage=("coverage", "max"),
        best_lane_mass_ratio=("lane_mass_ratio", "max"),
        min_reconstruction_mse=("reconstruction_mse", "min"),
        max_dead_features=("dead_features", "max"),
        max_redundant_valid_features=("redundant_valid_features", "max"),
    )
    .reset_index()
)

method_summary.to_csv("data/method_comparison_summary.csv", index=False)
print("Saved: data/method_comparison_summary.csv")
method_summary

## 10. Interpretation

Notebook 05 synthesizes prior outputs into a shared phase diagram.

Paper-level claim:

> Across matched residue-manifold data, reconstruction error, capacity, and structural recovery separate into distinct regimes. Compact NMF recovery occupies a high-coverage/high-alignment region, while sparse autoencoder configurations can occupy partial, fragmented, or diluted regimes even with increased capacity.

This prepares Notebook 06, where the provisional structure-quality score can be refined into a CGCS-style metric.

## 11. Optional download cell

Uncomment the final two lines to trigger a Colab download.

In [ ]:
# --- Optional: Download outputs (uncomment last line to trigger) ---

import os
import zipfile

zip_name = "05_coverage_phase_diagram_outputs.zip"
folders_to_zip = ["data", "figures"]

with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as z:
    for folder in folders_to_zip:
        if os.path.exists(folder):
            for root, _, filenames in os.walk(folder):
                for filename in filenames:
                    path = os.path.join(root, filename)
                    z.write(path, arcname=path)

print(f"Prepared: {zip_name}")

# from google.colab import files
# files.download(zip_name)